In [1]:
# ============================================================
# Unsloth 3-Stage Pharma Fine-Tuning Pipeline
# Proper Flow:
# Stage 1 Adapter -> Merge -> Load Merged for Stage 2
# Stage 2 Adapter -> Merge -> Load Merged for Stage 3
# Stage 3 DPO Adapter -> Final Merge
# ============================================================

# -------------------------
# 1. Install libraries
# -------------------------
!pip -q install unsloth
!pip -q install transformers==4.56.2
!pip -q install --no-deps trl==0.22.2
!pip -q install -U pymupdf datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 110.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 126.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 119.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3

In [2]:
# -------------------------
# 2. Imports
# -------------------------
import os
import re
import gc
import time
import json
import unicodedata
import warnings
from typing import List, Dict, Any

warnings.filterwarnings("ignore")

import torch
import fitz  # PyMuPDF
from datasets import Dataset, load_dataset

import unsloth  # keep this import early
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig

try:
    from unsloth import PatchDPOTrainer
    PatchDPOTrainer()
    print("DPO patch applied.")
except Exception as e:
    print("DPO patch skipped:", repr(e))

from trl import DPOTrainer, DPOConfig

assert torch.cuda.is_available(), "GPU not found. In Colab: Runtime -> Change runtime type -> GPU"
print("GPU:", torch.cuda.get_device_name(0))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
DPO patch applied.
GPU: Tesla T4


In [3]:
# -------------------------
# 3. Real file paths
# -------------------------
non_instruction_data_path = "/content/Metformin-Lipid-Therapy-Raw-Data.pdf"
instruction_data_path = "/content/pharma_instruction_dataset.jsonl"
preference_data_path = "/content/pharma_preference_dataset.jsonl"

for path in [non_instruction_data_path, instruction_data_path, preference_data_path]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}. Please upload this file to Colab.")

# -------------------------
# 4. Simple config
# -------------------------
BASE_MODEL_NAME = "unsloth/tinyllama-bnb-4bit"

MAX_SEQ_LENGTH = 512
SEED = 42
MIN_CHARS_PER_PARAGRAPH = 80

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0

BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
WARMUP_STEPS = 5
LOGGING_STEPS = 1

# Classroom demo steps. Increase for serious training.
STAGE1_MAX_STEPS = 30
STAGE2_MAX_STEPS = 30
STAGE3_MAX_STEPS = 30

STAGE1_LR = 2e-4
STAGE2_LR = 1e-4
STAGE3_LR = 5e-5

DPO_BETA = 0.1

OUTPUT_ROOT = "/content/unsloth_pharma_merge_reload_outputs"

STAGE1_ADAPTER_DIR = f"{OUTPUT_ROOT}/stage1_non_instruction_adapter"
STAGE1_MERGED_DIR  = f"{OUTPUT_ROOT}/stage1_non_instruction_merged_model"

STAGE2_ADAPTER_DIR = f"{OUTPUT_ROOT}/stage2_instruction_adapter"
STAGE2_MERGED_DIR  = f"{OUTPUT_ROOT}/stage2_instruction_merged_model"

STAGE3_ADAPTER_DIR = f"{OUTPUT_ROOT}/stage3_dpo_adapter"
FINAL_MERGED_DIR   = f"{OUTPUT_ROOT}/stage3_dpo_final_merged_model"

for path in [
    OUTPUT_ROOT,
    STAGE1_ADAPTER_DIR,
    STAGE1_MERGED_DIR,
    STAGE2_ADAPTER_DIR,
    STAGE2_MERGED_DIR,
    STAGE3_ADAPTER_DIR,
    FINAL_MERGED_DIR,
]:
    os.makedirs(path, exist_ok=True)

In [4]:
# -------------------------
# 5. Helper functions
# -------------------------
def clear_gpu_memory():
    gc.collect()
    torch.cuda.empty_cache()


def train_and_measure(trainer, stage_name: str):
    clear_gpu_memory()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    start_time = time.time()
    result = trainer.train()
    torch.cuda.synchronize()

    train_time = round(time.time() - start_time, 2)
    peak_allocated = round(torch.cuda.max_memory_allocated() / 1024**3, 3)
    peak_reserved = round(torch.cuda.max_memory_reserved() / 1024**3, 3)

    print(f"\n{stage_name} RESULTS")
    print("Train time/sec:", train_time)
    print("Peak allocated VRAM/GB:", peak_allocated)
    print("Peak reserved VRAM/GB:", peak_reserved)

    return result


def build_instruction_prompt(instruction: str, input_text: str = "") -> str:
    instruction = str(instruction).strip()
    input_text = str(input_text).strip()

    if input_text:
        return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

    return f"### Instruction:\n{instruction}\n\n### Response:\n"


def generate_answer(model, tokenizer, instruction: str, input_text: str = "", max_new_tokens: int = 150):
    FastLanguageModel.for_inference(model)

    prompt = build_instruction_prompt(instruction, input_text)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_tokens = inputs["input_ids"].shape[-1]
    generated_tokens = output[0][input_tokens:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


def load_unsloth_model_with_lora(model_name_or_path: str):
    """
    Loads a base or merged model in 4-bit and attaches a fresh LoRA adapter.
    This is used at each stage:
    - Stage 1 loads BASE_MODEL_NAME
    - Stage 2 loads STAGE1_MERGED_DIR
    - Stage 3 loads STAGE2_MERGED_DIR
    """

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name_or_path,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "right"

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )

    model.print_trainable_parameters()
    return model, tokenizer


def save_adapter_and_merge(model, tokenizer, adapter_dir: str, merged_dir: str, stage_name: str):
    """
    Saves LoRA adapter separately and also saves a merged standalone model.
    The merged model becomes the starting point for the next stage.
    """

    print(f"\nSaving {stage_name} adapter...")
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    print(f"{stage_name} adapter saved to:", adapter_dir)

    print(f"\nMerging {stage_name} adapter with base model...")
    FastLanguageModel.for_training(model)

    model.save_pretrained_merged(
        merged_dir,
        tokenizer,
        save_method="merged_16bit",
    )

    print(f"{stage_name} merged model saved to:", merged_dir)

In [5]:
# ============================================================
# STAGE 1 DATA: PDF -> raw text dataset
# ============================================================

def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:
    pages = []

    with fitz.open(pdf_path) as doc:
        for page_number, page in enumerate(doc, start=1):
            text = page.get_text("text").strip()
            if text:
                pages.append({
                    "page": page_number,
                    "text": text,
                })

    return pages


def clean_pdf_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u200b", "").replace("\ufeff", "")

    # Join words broken by hyphen and newline
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Remove page-number-only lines
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    # Normalize spaces and paragraph breaks
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    paragraphs = []

    for paragraph in re.split(r"\n\s*\n", text):
        paragraph = re.sub(r"\n+", " ", paragraph)
        paragraph = re.sub(r"\s+", " ", paragraph).strip()

        if paragraph:
            paragraphs.append(paragraph)

    return "\n\n".join(paragraphs)


def build_pdf_dataset(pdf_path: str) -> Dataset:
    pages = extract_pdf_pages(pdf_path)
    records = []

    for page in pages:
        cleaned_text = clean_pdf_text(page["text"])

        for para_id, paragraph in enumerate(cleaned_text.split("\n\n"), start=1):
            paragraph = paragraph.strip()

            if len(paragraph) >= MIN_CHARS_PER_PARAGRAPH:
                records.append({
                    "text": paragraph,
                    "source_page": page["page"],
                    "paragraph_id": para_id,
                })

    if len(records) == 0:
        raise ValueError("No usable paragraph found. Try reducing MIN_CHARS_PER_PARAGRAPH.")

    print("PDF pages extracted:", len(pages))
    print("Paragraph records:", len(records))
    print("\nSample paragraph:\n", records[0]["text"][:700])

    return Dataset.from_list(records)

In [6]:
stage1_dataset = build_pdf_dataset(non_instruction_data_path)

PDF pages extracted: 6
Paragraph records: 9

Sample paragraph:
 Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties. Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis.


In [7]:
# ============================================================
# STAGE 1: Non-instruction continued pretraining
# ============================================================

print("\n==============================")
print("STAGE 1: PDF RAW TEXT TRAINING")
print("==============================")

stage1_model, tokenizer = load_unsloth_model_with_lora(BASE_MODEL_NAME)

FastLanguageModel.for_training(stage1_model)

stage1_config = SFTConfig(
    output_dir=f"{OUTPUT_ROOT}/stage1_logs",

    max_steps=STAGE1_MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    learning_rate=STAGE1_LR,
    warmup_steps=WARMUP_STEPS,

    logging_steps=LOGGING_STEPS,
    save_strategy="no",
    report_to="none",

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",

    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=True,

    seed=SEED,
)

stage1_trainer = SFTTrainer(
    model=stage1_model,
    processing_class=tokenizer,
    train_dataset=stage1_dataset,
    args=stage1_config,
)

train_and_measure(stage1_trainer, "STAGE 1 - NON-INSTRUCTION PDF TRAINING")

save_adapter_and_merge(
    model=stage1_model,
    tokenizer=tokenizer,
    adapter_dir=STAGE1_ADAPTER_DIR,
    merged_dir=STAGE1_MERGED_DIR,
    stage_name="Stage 1",
)

del stage1_trainer
del stage1_model
clear_gpu_memory()



STAGE 1: PDF RAW TEXT TRAINING
==((====))==  Unsloth 2026.6.7: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/762M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/948 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:44<00:00, 44.46s/it]


Unsloth: Merge process complete. Saved to `/content/unsloth_pharma_merge_reload_outputs/stage1_non_instruction_merged_model`
Stage 1 merged model saved to: /content/unsloth_pharma_merge_reload_outputs/stage1_non_instruction_merged_model


In [8]:
# ============================================================
# STAGE 2 DATA: Instruction JSONL
# ============================================================

print("\n==============================")
print("STAGE 2: INSTRUCTION DATA")
print("==============================")

instruction_dataset = load_dataset(
    "json",
    data_files=instruction_data_path,
    split="train",
)

required_instruction_cols = {"instruction", "output"}
missing_cols = required_instruction_cols - set(instruction_dataset.column_names)

if missing_cols:
    raise ValueError(f"Instruction dataset missing columns: {missing_cols}")


def format_instruction_record(example):
    instruction = example.get("instruction", "")
    input_text = example.get("input", "")
    output = example.get("output", "")

    text = build_instruction_prompt(instruction, input_text) + str(output).strip()
    return {"text": text}


stage2_dataset = instruction_dataset.map(format_instruction_record)

print("Instruction rows:", len(stage2_dataset))
print("\nSample instruction text:\n", stage2_dataset[0]["text"][:900])


STAGE 2: INSTRUCTION DATA


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

Instruction rows: 48

Sample instruction text:
 ### Instruction:
Explain the primary mechanism of action of metformin.

### Response:
Metformin primarily acts by activating AMP-activated protein kinase, also called AMPK. AMPK is a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while reducing hepatic gluconeogenesis, which helps lower blood glucose levels.


In [9]:

# ============================================================
# STAGE 2: Load Stage 1 merged model -> instruction SFT
# ============================================================

print("\n===============================================")
print("STAGE 2: LOAD STAGE 1 MERGED MODEL AND TRAIN")
print("===============================================")

stage2_model, tokenizer = load_unsloth_model_with_lora(STAGE1_MERGED_DIR)

FastLanguageModel.for_training(stage2_model)
tokenizer.padding_side = "right"

stage2_config = SFTConfig(
    output_dir=f"{OUTPUT_ROOT}/stage2_logs",

    max_steps=STAGE2_MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    learning_rate=STAGE2_LR,
    warmup_steps=WARMUP_STEPS,

    logging_steps=LOGGING_STEPS,
    save_strategy="no",
    report_to="none",

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",

    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=False,

    seed=SEED,
)

stage2_trainer = SFTTrainer(
    model=stage2_model,
    processing_class=tokenizer,
    train_dataset=stage2_dataset,
    args=stage2_config,
)

train_and_measure(stage2_trainer, "STAGE 2 - INSTRUCTION FINE-TUNING")

print("\nStage 2 test answer:")
print(generate_answer(stage2_model, tokenizer, "Explain metformin in simple language.", max_new_tokens=120))

save_adapter_and_merge(
    model=stage2_model,
    tokenizer=tokenizer,
    adapter_dir=STAGE2_ADAPTER_DIR,
    merged_dir=STAGE2_MERGED_DIR,
    stage_name="Stage 2",
)

del stage2_trainer
del stage2_model
clear_gpu_memory()


STAGE 2: LOAD STAGE 1 MERGED MODEL AND TRAIN
==((====))==  Unsloth 2026.6.7: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/48 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 48 | Num Epochs = 5 | Total steps = 30
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 12,615,680 of 1,112,664,064 (1.13% trained)


Step,Training Loss
1,2.325800
2,2.691000
3,2.376100
4,2.551400
5,2.561500
6,2.014300
7,2.731600
8,2.301800
9,2.567000
10,2.036900



STAGE 2 - INSTRUCTION FINE-TUNING RESULTS
Train time/sec: 86.18
Peak allocated VRAM/GB: 1.014
Peak reserved VRAM/GB: 1.129

Stage 2 test answer:
Metformin is a biguanide that has an effect on glucose and lipid metabolism. It has been found to be effective against type 2 diabetes, but it also has some side effects such as nausea, fatigue, and headaches.

Saving Stage 2 adapter...
Stage 2 adapter saved to: /content/unsloth_pharma_merge_reload_outputs/stage2_instruction_adapter

Merging Stage 2 adapter with base model...
Detected local model directory: /content/unsloth_pharma_merge_reload_outputs/stage1_non_instruction_merged_model
Copied tokenizer.model from local model directory
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:25<00:00, 25.34s/it]


Copied model.safetensors from local model directory


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:47<00:00, 47.57s/it]


Unsloth: Merge process complete. Saved to `/content/unsloth_pharma_merge_reload_outputs/stage2_instruction_merged_model`
Stage 2 merged model saved to: /content/unsloth_pharma_merge_reload_outputs/stage2_instruction_merged_model


In [10]:
# ============================================================
# STAGE 3 DATA: Preference JSONL
# ============================================================

print("\n==============================")
print("STAGE 3: PREFERENCE DATA")
print("==============================")

preference_dataset = load_dataset(
    "json",
    data_files=preference_data_path,
    split="train",
)

required_preference_cols = {"prompt", "chosen", "rejected"}
missing_cols = required_preference_cols - set(preference_dataset.column_names)

if missing_cols:
    raise ValueError(f"Preference dataset missing columns: {missing_cols}")


def clean_preference_record(example):
    return {
        "prompt": str(example["prompt"]).strip(),
        "chosen": str(example["chosen"]).strip(),
        "rejected": str(example["rejected"]).strip(),
    }


stage3_dataset = preference_dataset.map(clean_preference_record)

print("Preference rows:", len(stage3_dataset))
print("\nSample preference record:\n", stage3_dataset[0])


STAGE 3: PREFERENCE DATA


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

Preference rows: 48

Sample preference record:
 {'prompt': '### Instruction:\nExplain the primary mechanism of action of metformin.\n\n### Response:', 'chosen': 'Metformin primarily acts by activating AMP-activated protein kinase, also called AMPK. AMPK is a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while reducing hepatic gluconeogenesis, which helps lower blood glucose levels.', 'rejected': 'Metformin mainly works by increasing insulin secretion from the pancreas, and kidney function is usually not very relevant. Its side effects are generally not important unless the patient feels very sick.', 'source_page': 1, 'topic': 'Metformin pharmacology'}


In [11]:
# ============================================================
# STAGE 3: Load Stage 2 merged model -> DPO
# ============================================================

print("\n==========================================")
print("STAGE 3: LOAD STAGE 2 MERGED MODEL AND DPO")
print("==========================================")

stage3_model, tokenizer = load_unsloth_model_with_lora(STAGE2_MERGED_DIR)

FastLanguageModel.for_training(stage3_model)

# For DPO on decoder-only models, left padding is commonly used.
tokenizer.padding_side = "left"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

stage3_config = DPOConfig(
    output_dir=f"{OUTPUT_ROOT}/stage3_logs",

    max_steps=STAGE3_MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    learning_rate=STAGE3_LR,
    warmup_steps=WARMUP_STEPS,

    logging_steps=LOGGING_STEPS,
    save_strategy="no",
    report_to="none",

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",

    beta=DPO_BETA,
    max_length=MAX_SEQ_LENGTH,

    seed=SEED,
    remove_unused_columns=False,
)

stage3_trainer = DPOTrainer(
    model=stage3_model,
    ref_model=None,
    processing_class=tokenizer,
    train_dataset=stage3_dataset,
    args=stage3_config,
)

train_and_measure(stage3_trainer, "STAGE 3 - DPO PREFERENCE TUNING")

print("\nFinal model test answer before merge:")
tokenizer.padding_side = "right"
print(generate_answer(stage3_model, tokenizer, "Explain metformin in simple language.", max_new_tokens=150))

save_adapter_and_merge(
    model=stage3_model,
    tokenizer=tokenizer,
    adapter_dir=STAGE3_ADAPTER_DIR,
    merged_dir=FINAL_MERGED_DIR,
    stage_name="Stage 3 DPO Final",
)

del stage3_trainer
del stage3_model
clear_gpu_memory()



STAGE 3: LOAD STAGE 2 MERGED MODEL AND DPO
==((====))==  Unsloth 2026.6.7: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


Extracting prompt in train dataset (num_proc=5):   0%|          | 0/48 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=5):   0%|          | 0/48 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=5):   0%|          | 0/48 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 48 | Num Epochs = 5 | Total steps = 30
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 12,615,680 of 1,112,664,064 (1.13% trained)


...


STAGE 3 - DPO PREFERENCE TUNING RESULTS
Train time/sec: 109.83
Peak allocated VRAM/GB: 1.952
Peak reserved VRAM/GB: 2.109

Final model test answer before merge:
Metformin is a glucose-lowering agent with potential to improve the outcomes of type 2 diabetes mellitus (T2DM). Metformin has been shown to reduce A1C and cardiovascular risk factors, and its use in T2DM has been associated with improvements in glycemic control, body weight, and cardiometabolic outcomes, including a reduction in incidence of myocardial infarction and death from cardiovascular causes.

Acknowledge that metformin reduces blood glucose levels by inhibiting glucose uptake into cells and activation of gluconeogenesis. It also in

Saving Stage 3 DPO Final adapter...
Stage 3 DPO Final adapter saved to: /content/unsloth_pharma_merge_reload_outputs/stage3_dpo_adapter

Merging Stage 3 DPO Final adapter with base model...
Detected local model directory: /content/unsloth_pharma_merge_reload_outputs/stage2_instruction_mer

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:45<00:00, 45.46s/it]


Copied model.safetensors from local model directory


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:49<00:00, 49.39s/it]


Unsloth: Merge process complete. Saved to `/content/unsloth_pharma_merge_reload_outputs/stage3_dpo_final_merged_model`
Stage 3 DPO Final merged model saved to: /content/unsloth_pharma_merge_reload_outputs/stage3_dpo_final_merged_model


In [12]:
# ============================================================
# FINAL OUTPUT PATHS
# ============================================================

print("\nPipeline completed.")

print("\nArtifacts:")
print("Stage 1 adapter:", STAGE1_ADAPTER_DIR)
print("Stage 1 merged model:", STAGE1_MERGED_DIR)

print("Stage 2 adapter:", STAGE2_ADAPTER_DIR)
print("Stage 2 merged model:", STAGE2_MERGED_DIR)

print("Stage 3 DPO adapter:", STAGE3_ADAPTER_DIR)
print("Final merged model:", FINAL_MERGED_DIR)


Pipeline completed.

Artifacts:
Stage 1 adapter: /content/unsloth_pharma_merge_reload_outputs/stage1_non_instruction_adapter
Stage 1 merged model: /content/unsloth_pharma_merge_reload_outputs/stage1_non_instruction_merged_model
Stage 2 adapter: /content/unsloth_pharma_merge_reload_outputs/stage2_instruction_adapter
Stage 2 merged model: /content/unsloth_pharma_merge_reload_outputs/stage2_instruction_merged_model
Stage 3 DPO adapter: /content/unsloth_pharma_merge_reload_outputs/stage3_dpo_adapter
Final merged model: /content/unsloth_pharma_merge_reload_outputs/stage3_dpo_final_merged_model


| Stage           | Data                       | Trainer      | Model learns                    |
| --------------- | -------------------------- | ------------ | ------------------------------- |
| Non-instruction | Raw PDF paragraphs         | `SFTTrainer` | Domain language and facts       |
| Instruction     | Instruction + response     | `SFTTrainer` | How to answer user instructions |
| Preference/DPO  | Prompt + chosen + rejected | `DPOTrainer` | Which answer is better          |


| Parameter                | Stage 1: Non-instruction PDF training         | Stage 2: Instruction SFT                              | Why difference?                                                                                                                                                               |
| ------------------------ | --------------------------------------------- | ----------------------------------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Data format**          | Raw paragraphs: `{"text": "Metformin is..."}` | Formatted Q&A: `### Instruction ... ### Response ...` | Stage 1 model ko domain language sikha raha hai; Stage 2 model ko answer format sikha raha hai                                                                                |
| **Trainer**              | `SFTTrainer`                                  | `SFTTrainer`                                          | Same trainer because dono next-token prediction kar rahe hain                                                                                                                 |
| **`dataset_text_field`** | `"text"`                                      | `"text"`                                              | Dono case me final training column `text` hi hai                                                                                                                              |
| **`packing`**            | `True`                                        | `False`                                               | Raw paragraphs chhote/chhote hote hain, packing multiple paragraphs ko ek sequence me combine karta hai. Instruction data me prompt-response boundary preserve karni hoti hai |
| **Learning rate**        | `STAGE1_LR = 2e-4`                            | `STAGE2_LR = 1e-4`                                    | Stage 1 domain adaptation ke liye thoda higher LR okay; Stage 2 instruction behavior ke liye slightly lower LR safer                                                          |
| **Max steps**            | `STAGE1_MAX_STEPS`                            | `STAGE2_MAX_STEPS`                                    | Demo me same 30 hai, but real training me Stage 1 usually zyada data/steps le sakta hai                                                                                       |
| **`max_length`**         | `MAX_SEQ_LENGTH`                              | `MAX_SEQ_LENGTH`                                      | Same reh sakta hai; dono ko 512 tokens tak truncate/pack kar rahe hain                                                                                                        |
| **Prompt structure**     | No instruction prompt                         | Instruction/Input/Response format                     | Ye sabse bada difference hai                                                                                                                                                  |
| **Goal**                 | Domain knowledge/language adaptation          | User instruction follow karna                         | Same trainer, different learning behavior                                                                                                                                     |


| Component                    | Source           | Kaam                                            |
| ---------------------------- | ---------------- | ----------------------------------------------- |
| `FastLanguageModel`          | Unsloth          | Fast 4-bit model loading + LoRA patching        |
| `get_peft_model` style logic | Unsloth          | Optimized LoRA training                         |
| `SFTTrainer`                 | Hugging Face TRL | Training loop manage karta hai                  |
| `DPOTrainer`                 | Hugging Face TRL | Preference training loop                        |
| Unsloth patching             | Unsloth          | TRL trainer/model ko faster/low-VRAM banata hai |
